# Phase 4: Domain-Adaptive Pre-Training (DAPT) on XLM-RoBERTa
### Dual-Variant Ablation Study on Sinhala-English Code-Mixed & Romanized Singlish

> **Project**: J26-SE-338 / Student Conversational Emotion & Mental Health Research (`c3-model-experiments`)  
> **Author**: C3 - Ramanayake R. H. B. D. G. (IT23164130) / Research Team  
> **Target Architecture**: `xlm-roberta-base` (125M parameters, 12 layers, 768 hidden dimension)  
> **Objective**: Masked Language Modeling (MLM) with dynamic 15% token masking  
> **Execution Target**: **Google Colab (Free Tier NVIDIA T4 GPU - 15GB VRAM)**  
> **Academic Foundation**: *Gururangan et al. ("Don't Stop Pretraining: Adapt Language Models to Domains and Tasks", ACL 2020)*  

---

### 🔬 Scientific Objectives & Mathematical Formulations

1. **Baseline Control ($PPL_0$)**: Establish the unadapted baseline perplexity of off-the-shelf `xlm-roberta-base` on the validation corpus prior to any weight updates:
   $$\mathcal{L}_{\text{MLM}}(\theta) = -\sum_{i \in M} \log P(x_i \mid \tilde{X}; \theta)$$
   $$PPL = \exp(\mathcal{L}_{\text{val}})$$

2. **Dual-Variant Ablation Study**:
   - **Variant 1 (`DAPT-Packed`)**: RoBERTa full-sentence packing up to `max_seq_length=256` separated by `</s></s>` delimiters (0% padding waste, maximum GPU utilization).
   - **Variant 2 (`DAPT-Dynamic`)**: Dynamic length-grouped batching (`group_by_length=True`) with strictly isolated sentences (zero cross-sentence attention interaction).

3. **Domain Acquisition Metrics**:
   $$\Delta PPL = PPL_{\text{base}} - PPL_{\text{DAPT}} \quad \text{and} \quad \% \text{ PPL Reduction} = \frac{PPL_{\text{base}} - PPL_{\text{DAPT}}}{PPL_{\text{base}}} \times 100\%$$


## Section 1: Hardware & Environment Verification
Verify NVIDIA GPU allocation (T4 with 15GB VRAM) and install modern Hugging Face libraries.

In [ ]:
# Check allocated GPU hardware
!nvidia-smi

# Install required research libraries
!pip install --quiet --upgrade transformers datasets evaluate accelerate matplotlib seaborn

In [ ]:
import datasets
import torch
import transformers

print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Transformers      : {transformers.__version__}")
print(f"Datasets          : {datasets.__version__}")

## Section 2: Workspace Setup & Repository Sync
Mount Google Drive or set up local working directory containing `data/adaptation/` partitions and `src/train_dapt_mlm.py`.

In [ ]:
import os

# 1. Create workspace directories
os.makedirs("src", exist_ok=True)
os.makedirs("data/adaptation", exist_ok=True)
os.makedirs("results/dapt", exist_ok=True)
os.makedirs("models", exist_ok=True)

# 2. If colab_dapt_bundle.zip is uploaded, extract it automatically
if os.path.isfile("colab_dapt_bundle.zip"):
    !unzip -q -o colab_dapt_bundle.zip
    print("Successfully extracted colab_dapt_bundle.zip!")
else:
    print("colab_dapt_bundle.zip not detected in /content. Please upload it.")

print("Working directory:", os.getcwd())
!ls -la src/ data/adaptation/

## Section 3: Data Integrity & Stratification Verification
Verify that both partitions created in Phase 3 are present and uncorrupted.

In [ ]:
train_file = "data/adaptation/dapt_train.txt"
val_file = "data/adaptation/dapt_val.txt"

with open(train_file, encoding="utf-8") as f:
    train_lines = [line.strip() for line in f if line.strip()]

with open(val_file, encoding="utf-8") as f:
    val_lines = [line.strip() for line in f if line.strip()]

print(f"Training partition   : {len(train_lines):,} sentences (~95.0%)")
print(f"Validation partition : {len(val_lines):,} sentences (~5.0%)")

# Preview sample sentences
print("\n--- Sample Validation Sentences ---")
for i, sent in enumerate(val_lines[:5], 1):
    print(f"{i}. {sent}")

## Section 4: Step 0 - Zero-Shot Baseline Control Benchmark ($PPL_0$)
Before beginning any continued pre-training, we evaluate off-the-shelf `xlm-roberta-base` on `dapt_val.txt` zero-shot to establish the baseline perplexity ($PPL_0$) and Top-1/Top-5 token prediction accuracy.

In [ ]:
!python src/train_dapt_mlm.py \
    --eval_baseline_only \
    --val_file data/adaptation/dapt_val.txt \
    --model_name_or_path xlm-roberta-base \
    --results_dir results/dapt \
    --max_seq_length 256 \
    --per_device_eval_batch_size 16

In [ ]:
import json

with open("results/dapt/baseline_evaluation.json", encoding="utf-8") as f:
    base_metrics = json.load(f)

print("==================================================")
print("VANILLA XLM-RoBERTa ZERO-SHOT BASELINE (PPL_0):")
print(f"  Model                 : {base_metrics['model_name_or_path']}")
print(f"  Validation Loss (L_0) : {base_metrics['eval_loss']:.4f}")
print(f"  Baseline Perplexity   : {base_metrics['perplexity']:.2f}")
print(f"  Top-1 Masked Token Acc: {base_metrics['top1_accuracy']:.2f}%")
print(f"  Top-5 Masked Token Acc: {base_metrics['top5_accuracy']:.2f}%")
print("==================================================")

## Section 5: Variant 1 Execution - `DAPT-Packed` (Strategy 1)
Trains Variant 1 with **full-sentence packing**: sentences are concatenated with `</s></s>` delimiters up to fixed 256-token buffers (0% padding waste, maximum GPU throughput).

In [ ]:
!python src/train_dapt_mlm.py \
    --strategy packed \
    --train_file data/adaptation/dapt_train.txt \
    --val_file data/adaptation/dapt_val.txt \
    --model_name_or_path xlm-roberta-base \
    --output_dir models/xlm-roberta-dapt-packed \
    --results_dir results/dapt \
    --max_seq_length 256 \
    --per_device_train_batch_size 8 \
    --per_device_eval_batch_size 8 \
    --gradient_accumulation_steps 8 \
    --learning_rate 2.5e-5 \
    --weight_decay 0.01 \
    --warmup_ratio 0.10 \
    --num_train_epochs 3.0 \
    --logging_steps 100 \
    --eval_steps 500 \
    --save_steps 500 \
    --early_stopping_patience 3

## Section 6: Variant 2 Execution - `DAPT-Dynamic` (Strategy 3)
Trains Variant 2 with **dynamic length-grouped batching**: sentences are grouped by length (`group_by_length=True`) with **strictly isolated conversational boundaries** (zero cross-sentence attention).

In [ ]:
!python src/train_dapt_mlm.py \
    --strategy dynamic \
    --train_file data/adaptation/dapt_train.txt \
    --val_file data/adaptation/dapt_val.txt \
    --model_name_or_path xlm-roberta-base \
    --output_dir models/xlm-roberta-dapt-dynamic \
    --results_dir results/dapt \
    --max_seq_length 256 \
    --per_device_train_batch_size 8 \
    --per_device_eval_batch_size 8 \
    --gradient_accumulation_steps 8 \
    --learning_rate 2.5e-5 \
    --weight_decay 0.01 \
    --warmup_ratio 0.10 \
    --num_train_epochs 3.0 \
    --logging_steps 100 \
    --eval_steps 500 \
    --save_steps 500 \
    --early_stopping_patience 3

## Section 7: Ablation Study Comparison & Empirical Analytics
Compile and compare baseline results against both DAPT variants.

In [ ]:
import pandas as pd

# Load all metric records
records = []
if os.path.isfile("results/dapt/baseline_evaluation.json"):
    with open("results/dapt/baseline_evaluation.json", encoding="utf-8") as f:
        base = json.load(f)
        records.append(
            {
                "Variant": "Vanilla XLM-R (Baseline)",
                "Strategy": "Zero-Shot Unadapted",
                "Val Loss": base["eval_loss"],
                "Perplexity (PPL)": base["perplexity"],
                "Δ PPL": 0.0,
                "% PPL Reduction": "0.0%",
                "Top-1 Acc (%)": base["top1_accuracy"],
                "Top-5 Acc (%)": base["top5_accuracy"],
                "Runtime": "-",
            }
        )

for strat in ["packed", "dynamic"]:
    mfile = f"results/dapt/metrics_{strat}.json"
    if os.path.isfile(mfile):
        with open(mfile, encoding="utf-8") as f:
            m = json.load(f)
            res = m["results"]
            records.append(
                {
                    "Variant": m["variant"],
                    "Strategy": "Packed (</s></s> buffer)"
                    if strat == "packed"
                    else "Dynamic Length-Grouped (Isolated)",
                    "Val Loss": res["final_eval_loss"],
                    "Perplexity (PPL)": res["final_perplexity"],
                    "Δ PPL": res.get("delta_perplexity", "-"),
                    "% PPL Reduction": f"{res.get('percent_perplexity_reduction', 0.0):.2f}%",
                    "Top-1 Acc (%)": res["top1_accuracy"],
                    "Top-5 Acc (%)": res["top5_accuracy"],
                    "Runtime": res.get("runtime_formatted", "-"),
                }
            )

df_comparison = pd.DataFrame(records)
display(df_comparison)

## Section 8: Publication-Quality Loss & Perplexity Trajectory Visualization
Generates high-resolution training dynamics plots comparing gradient descent and perplexity convergence.

In [ ]:
import math
import os

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for strat, color, label in [
    ("packed", "#1f77b4", "Variant 1: DAPT-Packed"),
    ("dynamic", "#2ca02c", "Variant 2: DAPT-Dynamic"),
]:
    mfile = f"results/dapt/metrics_{strat}.json"
    if os.path.isfile(mfile):
        with open(mfile, encoding="utf-8") as f:
            hist = json.load(f).get("training_loss_history", [])

        train_steps = [log["step"] for log in hist if "loss" in log]
        train_losses = [log["loss"] for log in hist if "loss" in log]
        eval_steps = [log["step"] for log in hist if "eval_loss" in log]
        eval_losses = [log["eval_loss"] for log in hist if "eval_loss" in log]
        eval_ppls = [math.exp(line) for line in eval_losses]

        # Subplot 1: Cross-Entropy Loss
        ax1.plot(
            train_steps,
            train_losses,
            label=f"{label} (Train Loss)",
            color=color,
            alpha=0.4,
            linestyle="--",
        )
        ax1.plot(
            eval_steps,
            eval_losses,
            label=f"{label} (Val Loss)",
            color=color,
            linewidth=2.5,
            marker="o",
        )

        # Subplot 2: Perplexity
        ax2.plot(
            eval_steps, eval_ppls, label=f"{label} (PPL)", color=color, linewidth=2.5, marker="s"
        )

# Add baseline horizontal line if available
if os.path.isfile("results/dapt/baseline_evaluation.json"):
    with open("results/dapt/baseline_evaluation.json", encoding="utf-8") as f:
        base_ppl = json.load(f).get("perplexity", None)
    if base_ppl:
        ax2.axhline(
            y=base_ppl,
            color="red",
            linestyle=":",
            linewidth=2,
            label=f"Vanilla Baseline PPL_0 ({base_ppl:.1f})",
        )

ax1.set_title("Training & Validation Cross-Entropy Loss (MLM)", fontsize=14, fontweight="bold")
ax1.set_xlabel("Training Optimization Steps", fontsize=12)
ax1.set_ylabel("MLM Cross-Entropy Loss", fontsize=12)
ax1.legend()

ax2.set_title("Validation Domain Perplexity Convergence (PPL)", fontsize=14, fontweight="bold")
ax2.set_xlabel("Training Optimization Steps", fontsize=12)
ax2.set_ylabel("Perplexity (exp(L_val))", fontsize=12)
ax2.legend()

plt.tight_layout()
plt.savefig("results/dapt/dapt_loss_curves.png", dpi=300, bbox_inches="tight")
plt.show()
print("[Plot Saved] High-resolution visualization saved to: results/dapt/dapt_loss_curves.png")

## Section 9: Qualitative Probe Predictions (Mask Infilling Comparison)
Inspect how baseline vs adapted models recover masked domain tokens in real code-mixed sentences.

In [ ]:
import json
import os


def load_qualitative(strat):
    fname = (
        f"results/dapt/metrics_{strat}.json"
        if strat != "baseline"
        else "results/dapt/baseline_evaluation.json"
    )
    if os.path.isfile(fname):
        with open(fname, encoding="utf-8") as f:
            data = json.load(f)
            return data.get("qualitative_probe_predictions", [])
    return []


base_probes = load_qualitative("baseline")
packed_probes = load_qualitative("packed")
dynamic_probes = load_qualitative("dynamic")

for i in range(min(5, len(base_probes))):
    sent = base_probes[i]["probe_sentence"]
    print(f"\n{'=' * 75}")
    print(f"PROBE [{i + 1}]: {sent}")
    print(f"{'=' * 75}")

    b_cands = [
        f"{c['token']} ({c['probability']:.2f})"
        for c in base_probes[i].get("top5_candidates", [])[:3]
    ]
    print("  Vanilla Baseline : ", ", ".join(b_cands))

    if i < len(packed_probes):
        p_cands = [
            f"{c['token']} ({c['probability']:.2f})"
            for c in packed_probes[i].get("top5_candidates", [])[:3]
        ]
        print("  DAPT-Packed      : ", ", ".join(p_cands))

    if i < len(dynamic_probes):
        d_cands = [
            f"{c['token']} ({c['probability']:.2f})"
            for c in dynamic_probes[i].get("top5_candidates", [])[:3]
        ]
        print("  DAPT-Dynamic     : ", ", ".join(d_cands))

## Section 10: Model Weight Export & Google Drive Backup
Compress and archive trained checkpoints for Stage B downstream emotion and mental health classification.

In [ ]:
# Compress models for convenient download or Google Drive backup
!tar -czf models/xlm-roberta-dapt-packed.tar.gz -C models xlm-roberta-dapt-packed
!tar -czf models/xlm-roberta-dapt-dynamic.tar.gz -C models xlm-roberta-dapt-dynamic
!tar -czf results/dapt_results.tar.gz -C results dapt

print("Archive archives created:")
!ls -lh models/*.tar.gz results/*.tar.gz

# If Google Drive is mounted, copy archives directly:
# !cp models/*.tar.gz /content/drive/MyDrive/J26-SE-338/models/
# !cp -r results/dapt /content/drive/MyDrive/J26-SE-338/results/